In [1]:
!pip install -q langchain-text-splitters tiktoken pypdf fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 30.7 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("rag_docs", exist_ok=True)

text = """Cloud Computing Fundamentals

Cloud computing provides computing resources such as servers, storage, and databases through network-based services. Organizations can scale resources according to demand.

Infrastructure as a Service provides virtualized computing resources. Platform as a Service gives developers an environment for building and deploying applications. Software as a Service delivers complete applications to end users.

Cloud systems can improve scalability and reduce the need for organizations to maintain physical infrastructure. Security, access control, monitoring, and cost management remain important considerations."""

with open("rag_docs/cloud_computing.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("Created sample document.")


Created sample document.


In [3]:
with open("rag_docs/cloud_computing.txt", "r", encoding="utf-8") as f:
    loaded_text = f.read()

print("Characters loaded:", len(loaded_text))
print(loaded_text[:500])


Characters loaded: 639
Cloud Computing Fundamentals

Cloud computing provides computing resources such as servers, storage, and databases through network-based services. Organizations can scale resources according to demand.

Infrastructure as a Service provides virtualized computing resources. Platform as a Service gives developers an environment for building and deploying applications. Software as a Service delivers complete applications to end users.

Cloud systems can improve scalability and reduce the need for or


In [4]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)

for paragraph in text.split("\n\n"):
    pdf.multi_cell(0, 8, paragraph)
    pdf.ln(3)

pdf.output("rag_docs/cloud_computing.pdf")
print("PDF created.")


PDF created.


In [5]:
from pypdf import PdfReader

reader = PdfReader("rag_docs/cloud_computing.pdf")
pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)

print("Pages:", len(reader.pages))
print(pdf_text[:500])


Pages: 1
Cloud Computing Fundamentals
Cloud computing provides computing resources such as servers, storage, and databases through
network-based services. Organizations can scale resources according to demand.
Infrastructure as a Service provides virtualized computing resources. Platform as a Service gives
developers an environment for building and deploying applications. Software as a Service delivers
complete applications to end users.
Cloud systems can improve scalability and reduce the need for organ


In [6]:
def fixed_chunks(text, size=220, overlap=40):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

fixed = fixed_chunks(loaded_text)

for i, chunk in enumerate(fixed, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)


--- Chunk 1 ---
Cloud Computing Fundamentals

Cloud computing provides computing resources such as servers, storage, and databases through network-based services. Organizations can scale resources according to demand.

Infrastructure as
--- Chunk 2 ---
 according to demand.

Infrastructure as a Service provides virtualized computing resources. Platform as a Service gives developers an environment for building and deploying applications. Software as a Service delivers c
--- Chunk 3 ---
ations. Software as a Service delivers complete applications to end users.

Cloud systems can improve scalability and reduce the need for organizations to maintain physical infrastructure. Security, access control, monit
--- Chunk 4 ---
ructure. Security, access control, monitoring, and cost management remain important considerations.


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(loaded_text)

for i, chunk in enumerate(recursive_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)


--- Chunk 1 ---
Cloud Computing Fundamentals

Cloud computing provides computing resources such as servers, storage, and databases through network-based services. Organizations can scale resources according to demand.
--- Chunk 2 ---
Infrastructure as a Service provides virtualized computing resources. Platform as a Service gives developers an environment for building and deploying applications
--- Chunk 3 ---
. Software as a Service delivers complete applications to end users.
--- Chunk 4 ---
Cloud systems can improve scalability and reduce the need for organizations to maintain physical infrastructure. Security, access control, monitoring, and cost management remain important considerations.


In [8]:
import re

def sentence_chunks(text, sentences_per_chunk=2):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [
        " ".join(sentences[i:i+sentences_per_chunk])
        for i in range(0, len(sentences), sentences_per_chunk)
    ]

sentence_based = sentence_chunks(loaded_text)

for i, chunk in enumerate(sentence_based, 1):
    print(f"Chunk {i}: {chunk}")


Chunk 1: Cloud Computing Fundamentals

Cloud computing provides computing resources such as servers, storage, and databases through network-based services. Organizations can scale resources according to demand.
Chunk 2: Infrastructure as a Service provides virtualized computing resources. Platform as a Service gives developers an environment for building and deploying applications.
Chunk 3: Software as a Service delivers complete applications to end users. Cloud systems can improve scalability and reduce the need for organizations to maintain physical infrastructure.
Chunk 4: Security, access control, monitoring, and cost management remain important considerations.


In [9]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def token_chunks(text, max_tokens=45, overlap=8):
    tokens = enc.encode(text)
    output = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        output.append(enc.decode(tokens[start:end]))
        start += max_tokens - overlap
    return output

token_based = token_chunks(loaded_text)

print("Token-based chunks:", len(token_based))
for i, chunk in enumerate(token_based, 1):
    print(f"--- Chunk {i} ---\n{chunk}")


Token-based chunks: 3
--- Chunk 1 ---
Cloud Computing Fundamentals

Cloud computing provides computing resources such as servers, storage, and databases through network-based services. Organizations can scale resources according to demand.

Infrastructure as a Service provides virtualized computing resources. Platform as a Service
--- Chunk 2 ---
ized computing resources. Platform as a Service gives developers an environment for building and deploying applications. Software as a Service delivers complete applications to end users.

Cloud systems can improve scalability and reduce the need for organizations to maintain physical infrastructure.
--- Chunk 3 ---
 need for organizations to maintain physical infrastructure. Security, access control, monitoring, and cost management remain important considerations.


In [10]:
import pandas as pd

comparison = pd.DataFrame({
    "Strategy": ["Fixed characters", "Recursive characters", "Sentences", "Tokens"],
    "Chunks": [len(fixed), len(recursive_chunks), len(sentence_based), len(token_based)]
})

print(comparison)


               Strategy  Chunks
0      Fixed characters       4
1  Recursive characters       4
2             Sentences       4
3                Tokens       3
